# Coffee Maker Reviews - LDA Topic Modeling (Positive Reviews)

## About This Lab

In this notebook, we use **Latent Dirichlet Allocation (LDA)** to discover hidden topics specifically within **positive** coffee maker reviews. By isolating positive reviews, we can discover what specific themes and qualities satisfied customers praise, separate from complaints.

This is a companion to the **LDA Topic Modeling NEG** notebook, which does the same analysis on negative reviews. Comparing the two reveals how customer language differs between satisfaction and dissatisfaction.

### What is LDA?

LDA is a statistical model that assumes each document is a mixture of topics, and each topic is a mixture of words. For example, a positive coffee maker review might be 60% about "brewing quality" and 40% about "ease of use." LDA discovers these topics automatically by analyzing which words tend to appear together across many documents.

### How Does Topic Modeling Differ from Classification?

| | Classification | Topic Modeling |
|---|---|---|
| **Type** | Supervised (needs labels) | Unsupervised (no labels needed) |
| **Goal** | Predict a category | Discover hidden themes |
| **Output** | One label per document | Mix of topics per document |
| **Example** | "This review is positive" | "This review is about brewing quality and ease of use" |

### Business Applications of Topic Modeling

- **Customer feedback analysis**: What themes keep appearing in customer reviews?
- **Market research**: What topics do people discuss about your product category?
- **Content organization**: Automatically tag and categorize large document collections.
- **Competitive analysis**: What are customers saying about competitors?

### What You Will Learn

- How to convert text into a document-term matrix for topic modeling
- How to train an LDA model on Amazon SageMaker
- How to extract and interpret the discovered topics
- How to visualize topics using word clouds, bar charts, and similarity heatmaps

### Dataset

We use a **positive-only subset** of the Coffee Maker Reviews dataset. The full dataset was preprocessed in the DATA lab (NLP - Data Preparation), then filtered to keep only reviews with `sentiment == "positive"`. The result is stored in `coffee_maker_binary_cleaned_POS.csv` with two key columns:

- **sentiment**: All rows are "positive" in this subset
- **cleaned_text**: The preprocessed review text (lowercased, stopwords removed, lemmatized)

# STEP 1: IMPORT LIBRARIES AND SETUP

In [ ]:
# Install any packages not included in the SageMaker environment
!pip install seaborn -q

In [ ]:
# Install any missing dependencies that are not included in the default SageMaker environment.
import subprocess, importlib
for package in ["seaborn", "wordcloud", "mxnet"]:
    if importlib.util.find_spec(package) is None:
        print(f"Installing {package}...")
        subprocess.check_call(["pip", "install", "-q", package])

# Standard Libraries
import os
import io
import time
import tarfile
import logging

# Data Handling & Visualization Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Scikit-learn Libraries for Text Vectorization and Similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# AWS & SageMaker Libraries for Model Training and Deployment
import boto3
import sagemaker
from sagemaker import Session, get_execution_role, image_uris
from sagemaker.estimator import Estimator
from sagemaker.amazon.common import write_spmatrix_to_sparse_tensor

# Additional Libraries
from botocore.exceptions import ClientError

# Apply patch for NumPy bool deprecation before importing MXNet.
# MXNet still uses the deprecated np.bool attribute (removed in NumPy 1.24+).
# This patch restores compatibility so MXNet can load model artifacts.
if not hasattr(np, "bool"):
    np.bool = bool

# Now import MXNet (used to load LDA model artifacts).
import mxnet as mx

# Logger Setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Global Variables
file_path = "coffee_maker_binary_cleaned_POS.csv"     # Path to the preprocessed dataset file
target_col = "sentiment"                          # Name of the sentiment column
text_col = "cleaned_text"                         # Name of the text column
endpoint_name = "lda-pos"                         # INSTRUCTION: Choose a unique endpoint name

print("All libraries imported successfully!")

# STEP 2: LOAD THE PREPROCESSED DATA

The text data has already been cleaned and preprocessed in the **DATA lab** (NLP - Data Preparation notebook). That notebook performed lowercasing, URL and HTML removal, emoji removal, tokenization, stopword removal, lemmatization, and length-based outlier removal.

This notebook uses `coffee_maker_binary_cleaned_POS.csv`, a **positive-reviews-only** subset created by filtering the full `coffee_maker_binary_cleaned.csv` to keep only rows where `sentiment == "positive"`.

**Before running this cell**, make sure `coffee_maker_binary_cleaned_POS.csv` is in the same directory as this notebook.

In [ ]:
def load_data(filepath):
    """
    Load data from a CSV file into a pandas DataFrame.

    This function reads the CSV file at the given filepath using pandas' read_csv
    method. It includes error handling for common issues, such as the file not being
    found. The error messages and exception re-raising help users diagnose problems
    during the loading process.

    Parameters:
        filepath (str): The path to the CSV file.

    Returns:
        pd.DataFrame: The DataFrame containing the loaded data.

    Raises:
        FileNotFoundError: If the file does not exist at the specified path.
        Exception: For any other error that occurs during the file reading.
    """
    try:
        # Attempt to read the CSV file into a DataFrame.
        data = pd.read_csv(filepath)
        return data
    except FileNotFoundError as e:
        # Inform the user that the file was not found and re-raise the exception.
        print(f"Error: The file at '{filepath}' was not found.")
        print(f"Make sure you have '{filepath}' in the same directory as this notebook.")
        raise e
    except Exception as e:
        # Handle any other exceptions that might occur during file reading.
        print(f"An error occurred while reading the file: {e}")
        raise e


# Load the preprocessed data.
df = load_data(file_path)
print("Data loaded successfully!\n")

# Drop any rows with missing text.
before = len(df)
df = df.dropna(subset=[text_col])
after = len(df)
if before != after:
    print(f"Dropped {before - after} rows with missing text.")

# Display the first few rows of the dataset to provide a quick visual overview.
print("First five rows of the dataset:")
print(df.head())

# Display the dimensions of the dataset (number of rows and columns).
print(f"\nShape of the dataset: {df.shape}")

# Display sentiment distribution.
print(f"\nSentiment Distribution:")
print(df[target_col].value_counts())

# STEP 3: VERIFY THE DATA

A quick check to confirm the preprocessed data loaded correctly. You already explored this dataset in detail during the DATA lab, so here we just verify the shape, sentiment distribution, and a few sample rows.

In [ ]:
# Quick data verification (full exploration was done in the DATA lab).
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Sentiment distribution.
print("Sentiment Distribution:")
sentiment_counts = df[target_col].value_counts()
for sentiment, count in sentiment_counts.items():
    print(f"  {target_col} = {sentiment}: {count} ({count / len(df):.1%})")

# Sample rows.
print("\n--- Sample Reviews ---")
for sentiment in sorted(df[target_col].unique()):
    sample = df[df[target_col] == sentiment].iloc[0]
    text = str(sample[text_col])
    preview = text[:120] + "..." if len(text) > 120 else text
    print(f"  {sentiment}: {preview}")

# STEP 4: VECTORIZE THE TEXT

LDA works with a **document-term matrix** (DTM), not raw text. The DTM is a table where:
- Each **row** represents a document (review)
- Each **column** represents a unique word from the vocabulary
- Each **cell** contains the count of how many times that word appears in that document

We use `CountVectorizer` from scikit-learn to build this matrix. The parameters control which words are included in the vocabulary:

- **max_features=1000**: Keep only the top 1,000 most frequent words.
- **min_df=2**: Ignore words that appear in fewer than 2 documents (too rare to be meaningful).
- **max_df=0.90**: Ignore words that appear in more than 90% of documents (too common to be distinctive).

In [ ]:
def vectorize_documents(texts, max_features=1000, min_df=2, max_df=0.90):
    """
    Convert a collection of text documents into a document-term matrix.

    This function uses scikit-learn's CountVectorizer to transform text into a sparse
    matrix of word counts. The vocabulary is filtered by frequency thresholds to keep
    only the most informative words.

    Parameters:
        texts (pd.Series): The text documents to vectorize.
        max_features (int): Maximum number of words to keep in the vocabulary. Defaults to 1000.
        min_df (int): Minimum number of documents a word must appear in. Defaults to 2.
        max_df (float): Maximum fraction of documents a word can appear in. Defaults to 0.90.

    Returns:
        tuple: (sparse_matrix, vocabulary, vectorizer)
            - sparse_matrix: The document-term matrix in sparse format.
            - vocabulary: Array of words in the vocabulary.
            - vectorizer: The fitted CountVectorizer object.
    """
    vectorizer = CountVectorizer(
        max_features=max_features,
        min_df=min_df,
        max_df=max_df
    )
    dtm = vectorizer.fit_transform(texts)
    vocabulary = vectorizer.get_feature_names_out()
    return dtm, vocabulary, vectorizer


# Vectorize the review text.
dtm, vocab, vectorizer = vectorize_documents(df[text_col])

# Display the document-term matrix shape.
print(f"Document-Term Matrix shape: {dtm.shape}")
print(f"  {dtm.shape[0]} documents (reviews)")
print(f"  {dtm.shape[1]} features (unique words in vocabulary)")

# Display a sample of the vocabulary.
print(f"\nSample vocabulary (first 20 words): {list(vocab[:20])}")
print(f"Sample vocabulary (last 20 words):  {list(vocab[-20:])}")

# STEP 5: PREPARE DATA AND UPLOAD TO S3

SageMaker's LDA algorithm expects data in **RecordIO protobuf** format, a compact binary format optimized for distributed training. We convert the sparse document-term matrix to this format and upload it to S3.

In [ ]:
# Initialize a SageMaker session.
sagemaker_session = sagemaker.Session()

# Get the IAM role associated with SageMaker.
role = get_execution_role()

# Define the S3 bucket and region for storing data.
bucket = sagemaker_session.default_bucket()  # Default bucket for SageMaker.
region = sagemaker_session.boto_region_name   # AWS region.
prefix = "sagemaker-lda/coffee-maker-pos"     # S3 prefix for positive reviews (unique to avoid collisions).

# Convert the sparse document-term matrix to RecordIO protobuf format.
print("Converting document-term matrix to RecordIO protobuf format...")
buffer = io.BytesIO()
write_spmatrix_to_sparse_tensor(buffer, dtm)
buffer.seek(0)

# Upload the protobuf file to S3.
s3_key = f"{prefix}/train/train_data.protobuf"
s3_client = boto3.client("s3")
s3_client.upload_fileobj(buffer, bucket, s3_key)

train_uri = f"s3://{bucket}/{s3_key}"
print(f"Training data uploaded to: {train_uri}")

# STEP 6: TRAIN THE LDA MODEL

Now we train the LDA model on SageMaker using only the minimum required hyperparameters.

- **num_topics**: The number of topics to discover (required, no default).
- **feature_dim**: The size of the vocabulary (required, must match the document-term matrix).
- **mini_batch_size**: How many documents to process at once during training (required, no default).

| Hyperparameter | Default Value |
|---|---|
| alpha0 | 1.0 |
| epochs | 1 |
| max_restarts | 10 |
| max_iterations | 1000 |

In [ ]:
# Retrieve the container image URI for the LDA algorithm.
container = image_uris.retrieve("lda", region)

# Determine the feature dimension (vocabulary size) from the document-term matrix.
feature_dim = dtm.shape[1]

# Instantiate the SageMaker Estimator for LDA.
lda_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=sagemaker_session
)

# Set hyperparameters for the LDA model (minimum required).
lda_estimator.set_hyperparameters(
    num_topics=3,                  # Number of topics to discover (required).
    feature_dim=feature_dim,       # Vocabulary size, must match the DTM (required).
    mini_batch_size=256            # Documents per mini-batch (required).
)

# Launch the model training process.
lda_estimator.fit({"train": train_uri})

print("Model training complete!")

# STEP 7: DEPLOY THE MODEL

Deploying the model creates a **real-time endpoint** that can assign topic distributions to new documents. Before deploying, we check if an endpoint with the same name already exists and delete it to avoid conflicts.

Note: We deploy the endpoint here so students can experiment with it, but for this lab the primary analysis comes from extracting the topic-word distributions directly from the model artifacts (STEP 8), which does not require a live endpoint.

In [ ]:
session = Session()                       # High-level SageMaker session.
sm_client = boto3.client("sagemaker")     # Low-level SageMaker client for endpoint management.

# Built-in waiter for polling endpoint status.
endpoint_in_service_waiter = sm_client.get_waiter("endpoint_in_service")


def delete_endpoint_and_config(endpoint_name: str, wait_for_deletion: bool = True) -> None:
    """
    Delete a SageMaker endpoint and its corresponding endpoint configuration if they exist.
    Optionally polls until resources are fully deleted.

    This function first checks if the endpoint exists. If it is in a transitional
    state (Creating or Updating), it waits for the endpoint to become InService
    before deleting. Then it deletes the endpoint configuration with the same name.

    Parameters:
        endpoint_name (str): The name of the endpoint to delete.
        wait_for_deletion (bool): Whether to poll until deletion is confirmed. Defaults to True.
    """
    # 1. Delete endpoint (if it exists).
    try:
        endpoint_desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
        endpoint_status = endpoint_desc["EndpointStatus"]

        # If the endpoint is Creating or Updating, wait for it to become InService before deleting.
        if endpoint_status in ("Creating", "Updating"):
            logger.info(f"Endpoint '{endpoint_name}' is in '{endpoint_status}' state. Waiting before delete.")
            endpoint_in_service_waiter.wait(EndpointName=endpoint_name)

        # Now delete the endpoint.
        logger.info(f"Deleting endpoint: {endpoint_name}")
        sm_client.delete_endpoint(EndpointName=endpoint_name)

    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Endpoint '{endpoint_name}' does not exist or has already been deleted.")
        else:
            raise e

    # 2. Delete endpoint config (if it exists).
    try:
        sm_client.describe_endpoint_config(EndpointConfigName=endpoint_name)
        logger.info(f"Deleting endpoint configuration: {endpoint_name}")
        sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Endpoint config '{endpoint_name}' does not exist or has already been deleted.")
        else:
            raise e

    # 3. Optionally poll for deletion.
    if wait_for_deletion:
        logger.info("Waiting for endpoint & configuration to be deleted...")
        for _ in range(30):
            endpoint_exists = True
            endpoint_config_exists = True

            try:
                sm_client.describe_endpoint(EndpointName=endpoint_name)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    endpoint_exists = False

            try:
                sm_client.describe_endpoint_config(EndpointConfigName=endpoint_name)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    endpoint_config_exists = False

            if not endpoint_exists and not endpoint_config_exists:
                logger.info("Endpoint and endpoint config fully deleted.")
                break

            logger.info("Endpoint or endpoint config still deleting... sleeping 10s.")
            time.sleep(10)
        else:
            logger.warning("Endpoint or endpoint config not fully deleted after 30 checks.")


# Delete any existing endpoint with the same name before deploying.
delete_endpoint_and_config(endpoint_name)

# Deploy the trained model as a real-time endpoint.
predictor = lda_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

logger.info(f"Endpoint '{endpoint_name}' deployed successfully and ready for inference.")

# STEP 8: EXTRACT TOPICS FROM MODEL

The trained LDA model contains a **beta matrix**: a table where each row is a topic and each column is a word from the vocabulary. The values represent how strongly each word is associated with each topic.

To access this matrix, we download the model artifact from S3, extract it, and load it using MXNet (the framework SageMaker uses internally for LDA).

In [ ]:
def extract_topics_from_model(estimator, vocab):
    """
    Extract the topic-word distribution matrix (beta) from a trained SageMaker LDA model.

    This function downloads the model artifact from S3, extracts the compressed archive,
    and loads the beta matrix using MXNet. The beta matrix has shape (num_topics, vocab_size)
    and represents how strongly each word is associated with each topic.

    Parameters:
        estimator: The trained SageMaker Estimator (must have a .model_data attribute).
        vocab (np.ndarray): The vocabulary array from the CountVectorizer.

    Returns:
        np.ndarray: The beta matrix of shape (num_topics, vocab_size).
    """
    # Parse the S3 location of the model artifact.
    model_data = estimator.model_data
    print(f"Model artifact location: {model_data}")
    bucket_name, key = model_data.replace("s3://", "").split("/", 1)

    # Create a local directory for the model files.
    local_model_dir = "lda_model"
    os.makedirs(local_model_dir, exist_ok=True)
    local_model_path = os.path.join(local_model_dir, "model.tar.gz")

    # Download the model archive from S3.
    print("Downloading model artifact from S3...")
    s3_download = boto3.client("s3")
    s3_download.download_file(bucket_name, key, local_model_path)

    # Extract the tar.gz archive.
    # The filter parameter is used for Python 3.12+ compatibility.
    print("Extracting model files...")
    with tarfile.open(local_model_path, "r:gz") as tar:
        file_list = tar.getnames()
        print(f"  Archive contents: {file_list}")
        try:
            tar.extractall(path=local_model_dir, filter="data")
        except TypeError:
            # Older Python versions do not support the filter parameter.
            tar.extractall(path=local_model_dir)

    # Locate the model file (typically named 'model_algo-1').
    model_files = [f for f in os.listdir(local_model_dir)
                   if f.startswith("model_") and os.path.isfile(os.path.join(local_model_dir, f))]
    if not model_files:
        raise FileNotFoundError("No model file found in the extracted archive.")

    model_file_path = os.path.join(local_model_dir, model_files[0])
    print(f"  Loading model from: {model_file_path}")

    # Load the model parameters using MXNet.
    model_params = mx.nd.load(model_file_path)

    # Extract the beta matrix (topic-word distributions).
    # The model contains [alpha, beta] where alpha is the topic prior and beta is the topic-word matrix.
    if isinstance(model_params, (list, tuple)):
        beta_nd = model_params[1] if len(model_params) >= 2 else model_params[0]
    elif isinstance(model_params, dict):
        values = list(model_params.values())
        beta_nd = values[1] if len(values) >= 2 else values[0]
    else:
        beta_nd = model_params

    beta = beta_nd.asnumpy()
    print(f"\nBeta matrix shape: {beta.shape} (topics x vocabulary)")
    print(f"  {beta.shape[0]} topics discovered")
    print(f"  {beta.shape[1]} words in vocabulary")

    return beta


# Extract the topic-word distributions from the trained model.
beta_matrix = extract_topics_from_model(lda_estimator, vocab)

## Print Top Words Per Topic

The most direct way to interpret a topic is to look at its highest-weighted words. Words with higher weights are more strongly associated with that topic.

In [ ]:
def print_top_words_per_topic(beta, vocab, num_top_words=15):
    """
    Print the highest-weighted words for each topic.

    For each topic, this function sorts the vocabulary by weight (highest first)
    and prints the top words along with their weights. Higher weights mean the
    word is more characteristic of that topic.

    Parameters:
        beta (np.ndarray): The topic-word distribution matrix of shape (num_topics, vocab_size).
        vocab (np.ndarray): The vocabulary array.
        num_top_words (int): Number of top words to display per topic. Defaults to 15.
    """
    print("\nTop Words Per Topic:")
    print("=" * 80)
    for topic_idx in range(beta.shape[0]):
        # Sort word indices by weight in descending order.
        top_indices = np.argsort(beta[topic_idx])[::-1][:num_top_words]

        # Get the corresponding words and weights.
        top_words = [vocab[i] for i in top_indices if i < len(vocab)]
        top_weights = [beta[topic_idx, i] for i in top_indices if i < len(vocab)]

        print(f"\nTopic {topic_idx}:")
        word_weight_pairs = [f"{w} ({wt:.4f})" for w, wt in zip(top_words, top_weights)]
        print("  " + ", ".join(word_weight_pairs))
    print("\n" + "=" * 80)


# Display the top words for each discovered topic.
print_top_words_per_topic(beta_matrix, vocab)

# STEP 9: VISUALIZE THE TOPICS

We create three types of visualizations to help interpret the discovered topics:

1. **Word clouds**: Show the most important words for each topic, with word size proportional to weight.
2. **Bar charts**: Show the exact weights of the top words for precise comparison.
3. **Similarity heatmap**: Show how similar or different the topics are from each other.

When interpreting topics, think about what business theme each topic might represent. For example, a topic with words like "coffee", "brew", "taste", "cup" might be about **brewing quality**, while a topic with "broken", "return", "warranty", "leak" might be about **product durability**.

## Word Clouds Per Topic

In [ ]:
# Generate a word cloud for each topic.
num_topics = beta_matrix.shape[0]
fig, axes = plt.subplots(num_topics, 1, figsize=(12, 4 * num_topics))

# Handle the edge case of a single topic.
if num_topics == 1:
    axes = [axes]

for topic_idx, ax in enumerate(axes):
    # Build a dictionary of word weights for this topic.
    word_weights = {}
    for i in range(len(vocab)):
        if i < beta_matrix.shape[1] and beta_matrix[topic_idx, i] > 0:
            word_weights[vocab[i]] = beta_matrix[topic_idx, i]

    # Generate the word cloud from the word weights.
    wc = WordCloud(width=800, height=300, background_color="white",
                   max_words=50, colormap="viridis",
                   prefer_horizontal=0.9, relative_scaling=0.5)
    wc.generate_from_frequencies(word_weights)

    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Topic {topic_idx}", fontsize=16)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Bar Charts of Top Words Per Topic

In [ ]:
# Create horizontal bar charts showing the top 10 words per topic.
num_top_words = 10
fig, axes = plt.subplots(num_topics, 1, figsize=(10, 4 * num_topics))

# Handle the edge case of a single topic.
if num_topics == 1:
    axes = [axes]

for topic_idx, ax in enumerate(axes):
    # Get the top words for this topic.
    top_indices = np.argsort(beta_matrix[topic_idx])[::-1][:num_top_words]
    top_words = [vocab[i] for i in top_indices]
    top_weights = beta_matrix[topic_idx, top_indices]

    # Sort by weight (ascending) for horizontal bar chart readability.
    sorted_idx = np.argsort(top_weights)
    top_words = [top_words[i] for i in sorted_idx]
    top_weights = top_weights[sorted_idx]

    # Create the bar chart.
    ax.barh(range(len(top_words)), top_weights, color="skyblue")
    ax.set_yticks(range(len(top_words)))
    ax.set_yticklabels(top_words)
    ax.set_title(f"Topic {topic_idx} - Top {num_top_words} Words")
    ax.set_xlabel("Weight")

plt.tight_layout()
plt.show()

## Topic Similarity Heatmap

The similarity heatmap shows how much overlap exists between topics. Values close to **1.0** mean two topics share many of the same important words (high overlap), while values close to **0.0** mean the topics are very different. Ideally, well-separated topics should have low similarity to each other.

In [ ]:
# Calculate cosine similarity between all pairs of topics.
similarity = cosine_similarity(beta_matrix)

# Create the heatmap.
plt.figure(figsize=(8, 6))
topic_labels = [f"Topic {i}" for i in range(num_topics)]
sns.heatmap(similarity, annot=True, fmt=".3f", cmap="viridis",
            xticklabels=topic_labels, yticklabels=topic_labels)
plt.title("Topic Similarity (Cosine Similarity)")
plt.tight_layout()
plt.show()

# Print interpretation guidance.
print("\nHow to interpret this heatmap:")
print("  - Diagonal values are always 1.0 (each topic is identical to itself).")
print("  - Off-diagonal values close to 1.0 mean two topics are very similar.")
print("  - Off-diagonal values close to 0.0 mean two topics are very different.")
print("  - If two topics are too similar, consider reducing the number of topics.")

# STEP 10: DELETE THE ENDPOINT (CLEANUP)

**WARNING: The deployed endpoint is running on a live EC2 instance and incurs charges as long as it is active. Always delete the endpoint when you are done to avoid unexpected AWS costs.**

This step asks for confirmation before deleting. If you choose not to delete now, remember to come back and delete the endpoint manually when you are finished experimenting.

In [ ]:
# Interactive confirmation before deleting the endpoint.
confirm = input("Do you want to delete the endpoint? (yes/no): ").strip().lower()

if confirm == "yes":
    delete_endpoint_and_config(endpoint_name)
    print(f"\nEndpoint '{endpoint_name}' has been deleted.")
else:
    print(f"\nEndpoint NOT deleted.")
    print("Remember to delete it manually when you are done to avoid ongoing AWS charges.")
    print(f"You can re-run this cell or call: delete_endpoint_and_config('{endpoint_name}')")